# Geometric Numerical Integration
## Störmer-Verlet (Velocity Verlet) Method

This notebook demonstrates geometric numerical integration using the Störmer-Verlet method,
applied to classical Hamiltonian systems.

### Key Properties Demonstrated:
1. **Symplecticity** - Preservation of phase space volume
2. **Time-reversibility** - Integration can be reversed exactly
3. **Long-term energy behaviour** - Bounded energy error (no secular drift)
4. **Splitting structure** - Kick-Drift-Kick decomposition

### Systems Studied:
- **Kepler Problem** - Planetary motion
- **Lennard-Jones** - Molecular dynamics

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Import our modules
import sys
sys.path.insert(0, '.')

from src.integrators import (
    verlet_step, euler_step, rk2_step, rk4_step, symplectic_euler_step,
    integrate, integrate_with_splitting_detail
)
from src.systems import KeplerSystem, LennardJonesSystem, HarmonicOscillator
from src.experiments import (
    run_kepler_comparison, long_term_energy_analysis,
    time_reversibility_test, symplectic_area_test,
    compute_local_error, lennard_jones_simulation
)
from src.plotting import (
    plot_kepler_orbits, plot_orbit_comparison, plot_rk4_verlet_contrast,
    plot_energy_error, plot_long_term_energy, plot_reversibility_test,
    plot_symplectic_area, plot_splitting_diagram, plot_splitting_flow,
    plot_lennard_jones, create_orbit_animation,
    plot_sv_vs_rk4_long_term, plot_sv_vs_rk2_euler_short_term,
    plot_energy_crossover, plot_symplectic_area_enhanced,
    create_exact_vs_sv_animation, create_sv_vs_rk2_animation,
    create_molecular_animation
)

# Plot settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['animation.embed_limit'] = 50000000  # 50MB limit for animations
%matplotlib inline

---
## 1. The Störmer-Verlet Integrator

The velocity Verlet method is a second-order symplectic integrator for Hamiltonian systems
of the form:

$$H(q,p) = T(p) + V(q) = \frac{1}{2}|p|^2 + V(q)$$

### Splitting Structure (Kick-Drift-Kick)

The method decomposes each step into three stages:

1. **KICK** (half-step momentum update): The force acts on the particle
   $$p_{1/2} = p_n - \frac{h}{2}\nabla V(q_n)$$

2. **DRIFT** (full-step position update): Free motion with updated momentum
   $$q_{n+1} = q_n + h \cdot p_{1/2}$$

3. **KICK** (half-step momentum update): Force acts again at new position
   $$p_{n+1} = p_{1/2} - \frac{h}{2}\nabla V(q_{n+1})$$

This symmetric splitting makes the method:
- **Time-reversible**: Running backward returns to the initial state
- **Symplectic**: Preserves phase space volume
- **Second-order accurate**: Local error is O(h³), global error is O(h²)

In [ ]:
# Visualize the kick-drift-kick splitting structure
# This diagram shows one complete Verlet step in phase space (q, p)
fig = plot_splitting_diagram()
plt.show()

**Understanding the diagram above:**

The diagram shows how a single Verlet step transforms the state $(q_n, p_n)$ to $(q_{n+1}, p_{n+1})$:

1. **Red arrow (KICK 1)**: Momentum changes vertically (force acts), position stays fixed
2. **Orange arrow (DRIFT)**: Position changes horizontally (free motion), momentum stays fixed  
3. **Purple arrow (KICK 2)**: Momentum changes again at the new position

The key insight is that each sub-step is *exactly solvable* - kicks and drifts are both simple operations. Composing them gives an approximation to the full Hamiltonian flow.

---
## 2. The Kepler Problem

The Kepler problem describes planetary motion under gravitational attraction:

$$V(q) = -\frac{\mu}{|q|}$$

$$\nabla V(q) = \frac{\mu \cdot q}{|q|^3}$$

The Hamiltonian is:
$$H(q,p) = \frac{1}{2}|p|^2 - \frac{\mu}{|q|}$$

This system conserves:
- Total energy $H$
- Angular momentum $L = q \times p$

In [ ]:
# Create Kepler system
kepler = KeplerSystem(mu=1.0)

# Initial conditions for elliptical orbit (eccentricity = 0.6)
q0, p0 = kepler.get_elliptical_orbit_ic(e=0.6, a=1.0)

print(f"Initial position: {q0}")
print(f"Initial momentum: {p0}")
print(f"Initial energy: {kepler.hamiltonian(q0, p0):.6f}")
print(f"Eccentricity: {kepler.eccentricity(q0, p0):.4f}")

### 2.1 Short-term Comparison: Störmer-Verlet vs RK2 vs Explicit Euler

We first compare three integrators over a **short time period** to show how quickly RK2 and Euler drift:
- **Verlet** (symplectic, 2nd order) - preserves geometric structure
- **RK2** (non-symplectic, 2nd order) - same order as Verlet but NOT symplectic
- **Euler** (non-symplectic, 1st order) - expect orbit degradation

**Key comparison: Verlet vs RK2** - Both are 2nd order methods, but only Verlet preserves energy!

In [ ]:
# Run comparison for 10 orbits (short term)
h = 0.01  # Time step
n_orbits = 10

kepler_results_short = run_kepler_comparison(kepler, q0, p0, h, n_orbits)

# Short-term comparison: SV vs RK2 vs Euler
fig = plot_sv_vs_rk2_euler_short_term(kepler_results_short)
plt.show()

**Observations:**
- **Verlet** maintains the closed elliptical orbit perfectly
- **RK2** starts drifting almost immediately - despite being the same order as Verlet!
- **Euler** spirals outward dramatically (energy increases unboundedly)

Even in this short time, the difference is clear. This demonstrates that **symplecticity matters more than order of accuracy**!

### 2.2 Long-term Comparison: Störmer-Verlet vs RK4

Now we compare Störmer-Verlet with RK4 over a **longer time period**. While RK4 appears good initially (it's 4th order!), over time its energy error grows unboundedly, eventually exceeding the bounded error of Verlet.

In [ ]:
# Run longer simulation to see RK4 drift
h_long = 0.02
n_orbits_long = 50  # More orbits to see RK4 drift

kepler_results_long = run_kepler_comparison(kepler, q0, p0, h_long, n_orbits=n_orbits_long)

# Long-term comparison: SV vs RK4
fig = plot_sv_vs_rk4_long_term(kepler_results_long)
plt.show()

**Key observation:** 
- **RK4** has higher per-step accuracy (4th order), so initially its error is smaller
- But over time, RK4's energy error **grows linearly** without bound
- **Verlet's** energy error remains **bounded and oscillating** for all time

This is the fundamental advantage of symplectic integrators for long-term Hamiltonian dynamics!

### 2.3 Energy Crossover: When RK4 Becomes Worse Than Verlet

Let's run an even longer simulation to clearly see the crossover point where RK4's cumulative error exceeds Verlet's bounded error.

In [ ]:
# Run very long simulation to see crossover
h_crossover = 0.03
n_orbits_crossover = 100  # Many orbits to see crossover

kepler_results_crossover = run_kepler_comparison(kepler, q0, p0, h_crossover, n_orbits=n_orbits_crossover)

# Energy crossover plot
fig = plot_energy_crossover(kepler_results_crossover)
plt.show()

**Critical insight:**

The crossover plot clearly shows that:
1. **RK4 starts better** (smaller error at short times due to O(h⁴) accuracy)
2. **Verlet stays bounded** (error oscillates around a fixed amplitude)
3. **RK4 eventually exceeds Verlet** (linear growth vs bounded oscillation)

The error ratio plot shows this even more dramatically - the ratio grows without bound!

---
## 3. Long-term Energy Behaviour: All Methods

Let's compare all five integrators to see the full picture:
- **Symplectic methods** (Verlet, Symplectic Euler): Bounded energy error
- **Non-symplectic methods** (Euler, RK2, RK4): Secular drift

In [ ]:
# Long-term energy analysis
T_final = 200  # Long integration time
h = 0.05

energy_results = long_term_energy_analysis(kepler, q0, p0, h, T_final)

# Plot energy errors
fig = plot_long_term_energy(energy_results)
plt.show()

# Print max errors
print("Maximum relative energy errors:")
for method in ['verlet', 'symplectic_euler', 'rk2', 'euler', 'rk4']:
    if method in energy_results:
        print(f"  {method:20s}: {energy_results[method]['max_error']:.2e}")

**Key observations:**

1. **Symplectic methods (Verlet, Symplectic Euler)**: Energy error oscillates but stays bounded. No secular growth.

2. **Non-symplectic methods (Euler, RK2, RK4)**: 
   - Euler: Energy grows explosively (visible in linear plot)
   - RK2: Energy drifts (same order as Verlet, but NOT symplectic!)
   - RK4: Energy drifts linearly (visible in log plot as slow growth)

**The RK2 vs Verlet comparison is crucial**: Both are 2nd order methods, but only Verlet preserves energy!

---
## 4. Time-Reversibility

The Verlet method is **time-reversible**: if we integrate forward N steps, reverse momenta, and integrate forward N steps again, we return to the initial state (up to round-off).

**Test procedure:**
1. Start at $(q_0, p_0)$
2. Integrate forward N steps → $(q_N, p_N)$
3. Reverse momenta: $(q_N, -p_N)$
4. Integrate forward N steps → $(q_{2N}, p_{2N})$
5. Measure error: $|q_{2N} - q_0|$

For time-reversible methods, this error is at machine precision. For non-reversible methods, error accumulates.

In [ ]:
# Time-reversibility test
step_sizes = np.logspace(-3, -1, 10)
n_steps = 100

reversibility_results = time_reversibility_test(
    kepler.gradV, q0, p0, step_sizes, n_steps
)

fig = plot_reversibility_test(reversibility_results)
plt.show()

**Observations:**

- **Verlet**: Shows very small errors (near machine precision ~10⁻¹⁴) - truly time-reversible
- **Symplectic Euler**: Shows O(h) errors - symplectic but NOT time-reversible
- **Euler**, **RK2**, and **RK4**: Show O(h) and O(h²) errors respectively - neither time-reversible nor symplectic

---
## 5. Symplectic Area Preservation

Symplectic integrators preserve the symplectic 2-form, which implies **conservation of phase space area** (Liouville's theorem).

**Test:** Take a circle of initial conditions in phase space, integrate forward, and measure the area:
- Symplectic methods preserve the area (ratio = 1.0)
- Non-symplectic methods distort it (ratio ≠ 1.0)

We use a harmonic oscillator for cleaner visualization and **larger step size/more steps** to show the distortion clearly for RK2 and RK4.

In [ ]:
# Use harmonic oscillator for cleaner visualization
ho = HarmonicOscillator(omega=1.0)

# Center of initial circle
center_q = np.array([1.0])
center_p = np.array([0.0])
radius = 0.3

# Use larger step size and more steps to show RK2 and RK4 distortion clearly
area_results = symplectic_area_test(
    ho.gradV, center_q, center_p,
    radius=radius, n_points=100, h=0.4, n_steps=40  # Larger h and more steps for visible distortion
)

# Enhanced area plot with clear distortion visualization
fig = plot_symplectic_area_enhanced(area_results)
plt.show()

print("\nArea ratios (should be 1.0 for symplectic methods):")
for method in ['verlet', 'symplectic_euler', 'rk2', 'euler', 'rk4']:
    if method in area_results:
        ratio = area_results[method]['area_ratio']
        status = "PRESERVED" if abs(ratio - 1.0) < 0.01 else "DISTORTED"
        print(f"  {method:20s}: {ratio:.6f} ({status})")

**Observations:**

- **Verlet & Symplectic Euler**: Area ratio ≈ 1.0 (PRESERVED) - The circle rotates but maintains its area
- **Euler**: Area dramatically expands (energy gain causes expansion in phase space)
- **RK2**: Area distorted - Despite being 2nd order like Verlet, it doesn't preserve symplectic structure!
- **RK4**: Area distorted - Despite high accuracy per step, it doesn't preserve the symplectic structure

**The RK2 comparison is key**: Same order as Verlet, but fails to preserve area!

---
## 6. Splitting Visualization on the Kepler Orbit

Let's visualize how the kick-drift-kick splitting works for actual Kepler orbit integration.

In [ ]:
# Get detailed splitting information
h_split = 0.3  # Large step for visibility
n_steps_split = 5

splitting_data = integrate_with_splitting_detail(
    q0, p0, h_split, n_steps_split, kepler.gradV
)

# Show the splitting for step 0
fig = plot_splitting_flow(splitting_data, step_idx=0)
plt.show()

In [ ]:
# Visualize multiple steps showing the full trajectory
fig, ax = plt.subplots(figsize=(10, 10))

# Plot central body (Sun)
ax.scatter([0], [0], s=400, c='yellow', marker='*', edgecolors='orange', 
           linewidth=2, zorder=10, label='Sun')

# Plot the position trajectory
q_traj = splitting_data['q']
ax.plot(q_traj[:, 0], q_traj[:, 1], 'b-', linewidth=2, alpha=0.6, label='Trajectory')

# Mark each complete step with different colors
colors = plt.cm.viridis(np.linspace(0, 1, n_steps_split + 1))
for i in range(n_steps_split + 1):
    ax.scatter(q_traj[i, 0], q_traj[i, 1], s=150, c=[colors[i]], 
               edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f'Step {i}', (q_traj[i, 0], q_traj[i, 1]), 
                xytext=(10, 10), textcoords='offset points', fontsize=9)

# Draw drift arrows for each step
for i in range(n_steps_split):
    q_start = splitting_data['q'][i]
    q_drift = splitting_data['q_after_drift'][i]
    
    ax.annotate('', xy=(q_drift[0], q_drift[1]), xytext=(q_start[0], q_start[1]),
               arrowprops=dict(arrowstyle='->', color='orange', lw=2, alpha=0.7))

ax.set_xlabel('x position', fontsize=12)
ax.set_ylabel('y position', fontsize=12)
ax.set_title(f'Verlet Integration: {n_steps_split} Steps on Kepler Orbit\n'
             f'(Step size h = {h_split}, Orange arrows show DRIFT phase)',
             fontsize=13)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

---
## 7. Lennard-Jones Molecular Dynamics

The Lennard-Jones potential models interatomic forces in noble gases and simple molecules:

$$V_{LJ}(r) = 4\varepsilon \left[ \left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6} \right]$$

- **$r^{-12}$ term**: Repulsive (Pauli exclusion at short range)
- **$r^{-6}$ term**: Attractive (van der Waals/London dispersion)
- **Equilibrium distance**: $r_{eq} = 2^{1/6}\sigma \approx 1.122\sigma$

In [ ]:
# Two-particle Lennard-Jones system
lj = LennardJonesSystem(n_particles=2, epsilon=1.0, sigma=1.0, dim=2)

# Initial conditions: particles separated, moving perpendicular to separation
q0_lj, p0_lj = lj.get_two_particle_ic(separation=1.5, velocity=0.3)

print(f"Initial positions (2 particles, 2D each):")
print(f"  Particle 1: {q0_lj.reshape(2, 2)[0]}")
print(f"  Particle 2: {q0_lj.reshape(2, 2)[1]}")
print(f"\nInitial separation: {np.linalg.norm(q0_lj.reshape(2, 2)[0] - q0_lj.reshape(2, 2)[1]):.3f}")
print(f"Equilibrium separation: {lj.sigma * 2**(1/6):.3f}")
print(f"Initial energy: {lj.hamiltonian(q0_lj, p0_lj):.6f}")

In [ ]:
# Run simulation
h_lj = 0.005  # Small step for stiff LJ forces
n_steps_lj = 10000

lj_results = lennard_jones_simulation(lj, q0_lj, p0_lj, h_lj, n_steps_lj)

# Plot results
fig = plot_lennard_jones(lj_results, lj)
plt.show()

In [ ]:
# Three-particle system: more complex dynamics
lj3 = LennardJonesSystem(n_particles=3, epsilon=1.0, sigma=1.0, dim=2)
q0_lj3, p0_lj3 = lj3.get_three_particle_ic(separation=1.5, velocity=0.2)

lj3_results = lennard_jones_simulation(lj3, q0_lj3, p0_lj3, h=0.002, n_steps=20000)

fig = plot_lennard_jones(lj3_results, lj3)
plt.show()

---
## 8. Local Error Estimation

We can estimate the local truncation error by comparing with the exact solution (for the harmonic oscillator) or using Richardson extrapolation.

**Expected orders:**
- Euler: O(h) - first order
- Verlet: O(h²) - second order
- RK4: O(h⁴) - fourth order

In [ ]:
# Use harmonic oscillator (has exact solution)
ho = HarmonicOscillator(omega=1.0)
q0_ho = np.array([1.0, 0.0])
p0_ho = np.array([0.0, 1.0])

# Test various step sizes
h_values = [0.1, 0.05, 0.025, 0.0125]

print("Local Error Estimates (Harmonic Oscillator):")
print("="*60)
print(f"{'h':>10s} {'Euler':>12s} {'Verlet':>12s} {'RK4':>12s}")
print("-"*60)

for h in h_values:
    errors = compute_local_error(ho.gradV, q0_ho, p0_ho, h, ho.exact_solution)
    print(f"{h:>10.4f} {errors['euler']['total_error']:>12.2e} "
          f"{errors['verlet']['total_error']:>12.2e} "
          f"{errors['rk4']['total_error']:>12.2e}")

print("-"*60)
print("Expected orders: Euler O(h), Verlet O(h²), RK4 O(h⁴)")

In [ ]:
# Verify convergence orders visually
h_array = np.array(h_values)
euler_errors = [compute_local_error(ho.gradV, q0_ho, p0_ho, h, ho.exact_solution)['euler']['total_error'] 
                for h in h_array]
verlet_errors = [compute_local_error(ho.gradV, q0_ho, p0_ho, h, ho.exact_solution)['verlet']['total_error'] 
                 for h in h_array]
rk4_errors = [compute_local_error(ho.gradV, q0_ho, p0_ho, h, ho.exact_solution)['rk4']['total_error'] 
              for h in h_array]

fig, ax = plt.subplots(figsize=(10, 7))

ax.loglog(h_array, euler_errors, 'ro-', label='Euler (measured)', linewidth=2, markersize=10)
ax.loglog(h_array, verlet_errors, 'go-', label='Verlet (measured)', linewidth=2, markersize=10)
ax.loglog(h_array, rk4_errors, 'bo-', label='RK4 (measured)', linewidth=2, markersize=10)

# Reference lines for expected orders
ax.loglog(h_array, h_array * euler_errors[0]/h_array[0], 'r--', alpha=0.5, linewidth=1.5, label='O(h) reference')
ax.loglog(h_array, h_array**2 * verlet_errors[0]/h_array[0]**2, 'g--', alpha=0.5, linewidth=1.5, label='O(h²) reference')
ax.loglog(h_array, h_array**4 * rk4_errors[0]/h_array[0]**4, 'b--', alpha=0.5, linewidth=1.5, label='O(h⁴) reference')

ax.set_xlabel('Step size h', fontsize=12)
ax.set_ylabel('Local Truncation Error', fontsize=12)
ax.set_title('Local Error vs Step Size\n(Slopes match expected convergence orders)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

---
## 9. Animations

Below are several focused animations demonstrating the behavior of different integrators.

### 9.1 Animation: Exact Orbit vs Störmer-Verlet

This animation shows how well the Störmer-Verlet method tracks the exact Kepler orbit.

In [ ]:
# Animation: Exact orbit vs Störmer-Verlet
anim_exact_sv = create_exact_vs_sv_animation(kepler_results_short, kepler, 
                                              interval=40, skip=5, max_frames=200)
HTML(anim_exact_sv.to_jshtml())

### 9.2 Animation: Störmer-Verlet vs RK2 (Both 2nd Order!)

This animation compares two 2nd order methods: Verlet (symplectic) and RK2 (non-symplectic). Watch how RK2 drifts while Verlet stays on track!

In [ ]:
# Animation: Störmer-Verlet vs RK2
anim_sv_rk2 = create_sv_vs_rk2_animation(kepler_results_short, kepler,
                                          interval=40, skip=5, max_frames=200)
HTML(anim_sv_rk2.to_jshtml())

### 9.3 Animation: Two-Particle Molecular Dynamics with Energy Conservation

This animation shows the Lennard-Jones two-particle dynamics with a real-time energy conservation plot.

In [ ]:
# Animation: Two-particle molecular dynamics
anim_lj2 = create_molecular_animation(lj_results, lj, 
                                       title="Lennard-Jones Two-Particle Dynamics",
                                       interval=50, skip=50, max_frames=200)
HTML(anim_lj2.to_jshtml())

### 9.4 Animation: Three-Particle Molecular Dynamics with Energy Conservation

This animation shows the more complex three-particle Lennard-Jones dynamics.

In [ ]:
# Animation: Three-particle molecular dynamics
anim_lj3 = create_molecular_animation(lj3_results, lj3,
                                       title="Lennard-Jones Three-Particle Dynamics",
                                       interval=50, skip=100, max_frames=200)
HTML(anim_lj3.to_jshtml())

---
## 10. Summary

### Properties of the Störmer-Verlet Method

| Property | Verlet | RK2 | Symplectic Euler | Euler | RK4 |
|----------|--------|-----|------------------|-------|-----|
| Order | 2 | 2 | 1 | 1 | 4 |
| Symplectic | Yes | **No** | Yes | No | No |
| Time-reversible | Yes | No | No | No | No |
| Energy error | Bounded | **Drifts** | Bounded | Grows | Drifts |
| Area preserving | Yes | **No** | Yes | No | No |

**Key insight from RK2 comparison:** Verlet and RK2 are both 2nd order methods, but Verlet preserves energy while RK2 does not. This proves that **symplecticity, not accuracy order, determines long-term energy behaviour**.

### Why Symplectic Integrators?

1. **Long-term stability**: Energy remains bounded, no artificial heating/cooling
2. **Geometric fidelity**: Preserves the structure of Hamiltonian mechanics
3. **Time-reversibility**: Important for statistical mechanics and hybrid Monte Carlo
4. **Efficiency**: Simple explicit formula, no iteration required

### Applications

- Molecular dynamics simulations
- Celestial mechanics and space mission design
- Particle accelerator design
- Quantum Monte Carlo
- Game physics engines

---
## References

1. Hairer, E., Lubich, C., & Wanner, G. (2006). *Geometric Numerical Integration*. Springer.
2. Leimkuhler, B., & Reich, S. (2004). *Simulating Hamiltonian Dynamics*. Cambridge University Press.
3. Verlet, L. (1967). Computer "Experiments" on Classical Fluids. *Physical Review*, 159(1), 98-103.

See the `docs/` folder for additional reference materials.